# Pure $\zeta(3)$: approximants and spectral constants

This independently runnable notebook supports **Theorem 6.16 (Irrationality measures)**, label `thm:z23-measures`, through **Appendix B.4**, label `app:zeta3-measure`. It also checks the constants used in **Lemma 6.8**, label `lem:z34-selected-error`. The limit identification is **Lemma 6.7**, label `lem:z34-limits`.

Run all cells in a fresh Python kernel. The companion files in this folder are required; no other notebook, original archive, internet connection, or Lean installation is needed. Dependencies are `ramanujantools`, `sympy`, and `python-flint` (the bundle requirements give exact tested versions).

**Scope.** We reconstruct the native CMF, verify exact finite partial-fraction/readout examples, reproduce the limiting characteristic polynomial, and enclose its spectral constants with rational arithmetic. The all-index identification with the Section 6 approximants is proved below by gamma reflection. Convergence and the error estimate then use the paper's joint analytic argument; finite examples do not replace it. No Lean code is created, changed, or executed.


In [1]:
from pathlib import Path
import sys, time, json, math
# Supports opening this notebook directly or running from the bundle root.
HERE = Path.cwd()
if not (HERE / 'approximants.py').exists():
    HERE = HERE / 'pure_zeta3'
assert (HERE / 'approximants.py').exists(), 'Run from this folder or the bundle root.'
sys.path.insert(0, str(HERE.resolve()))
if hasattr(sys, 'set_int_max_str_digits'):
    sys.set_int_max_str_digits(0)
started = time.perf_counter()
from fractions import Fraction as F
from flint import fmpq, arb, ctx
import sympy as sp
from approximants import partial_fractions, partial_fractions_direct, kernel_readout, finite_integer_check
from rates import native_trajectory, matrix_at_infinity, spectral_enclosures


## 1. Keep the two normalizations distinct

The section uses forward $q_n,p_{3,n}$ with $q_n=12F_n$ and the hypergeometric ${}_5F_4$ trajectory. Appendix B.4 uses capital $Q_n,P_{3,n}$ for the opposite-direction native Meijer $G$ trajectory:
\[
(1,1,0,0,0;0,0,0,0,0),\qquad
(16,17,-11,-12,-10;2,1,0,-1,-2).
\]
Their **ratios** agree: $p_{3,n}/q_n=P_{3,n}/Q_n$. Do not silently identify the raw sequences or the section's binomial factor $C_n$ with the appendix's $C'_n$.

Write $W_n=M(0)\cdots M(n-1)$ for this native trajectory. Its readout is
\[
Q_n=(-1)^n(W_n)_{00},\qquad
P_{3,n}=(-1)^n\bigl(-(W_n)_{20}/2+(W_n)_{30}/4\bigr).
\]
Thus the common sign cancels in the ratio. Here $n$ always means the number of steps; the source archive sometimes used $N=n+1$.


In [2]:
M = native_trajectory()  # Rebuilt by ramanujantools, with no cached matrix.
assert M.shape == (4, 4)
n_symbol = sp.Symbol('n')
walks = [sp.eye(4)] + list(M.walk({n_symbol: 1}, [1, 2, 3], {n_symbol: 0}))
print('Native Meijer G trajectory rebuilt; matrix dimension:', M.shape)


Native Meijer G trajectory rebuilt; matrix dimension: (4, 4)


## 2. Appendix B.4, “The approximations”: exact partial fractions

To avoid confusing the paper's two uses of $F_n$, Python calls the rational kernel `kernel`:
\[
F_n(t)=\frac{\prod_{u=0}^2(t-(10+u)n)_{10n}}
{(t+n)_{16n+1}(t+2n)_{14n+1}}
=\sum_{j=n}^{17n}\left(\frac{a_j}{(t+j)^2}+\frac{b_j}{t+j}\right).
\]
At a double pole, $a_j=\lim_{t\to-j}(t+j)^2F_n(t)$ and $b_j$ is the derivative of this regularized function. Logarithmic differentiation expresses $b_j/a_j$ as a sum of reciprocal linear factors. At a simple pole $a_j=0$. The module provides both the direct product calculation and its faster factorial/harmonic version, and the next cell compares them exactly.

With $H_m^{(r)}=\sum_{k=1}^m k^{-r}$ and $H_0^{(r)}=0$,
\[
Q_n=\sum_j a_j,\qquad
P_{3,n}=\sum_j\left(a_jH_{j+2n}^{(3)}+\tfrac12b_jH_{j+2n}^{(2)}\right).
\]
These are rational numbers. Their integrality requires the normalization in the next section. The native-walk identities checked at $n=0,1,2,3$ are **finite regression checks**. The following separate argument identifies the appendix sums with the Section 6 approximants for every index.


In [3]:
for n, W in enumerate(walks):
    assert partial_fractions_direct(n) == partial_fractions(n)
    Q_n, P3_n, _ = kernel_readout(n)
    exact = lambda x: fmpq(str(x))
    assert Q_n == (-1)**n * exact(W[0, 0])
    assert P3_n == (-1)**n * (-exact(W[2, 0])/2 + exact(W[3, 0])/4)
    assert sum((b for a, b in partial_fractions(n).values()), fmpq(0)) == 0
print('Exact partial fractions and native CMF readouts agree at n = 0, 1, 2, 3.')


Exact partial fractions and native CMF readouts agree at n = 0, 1, 2, 3.


### Same approximants at every index — Appendix B.4 and Lemmas 6.9–6.10

This is the identification used in the pure case of Theorem 6.16. It uses the kernel index $n$ on both sides; the separate native walk convention does not introduce a shift here.

Set $C'_n=(15n)!^2/(10n)!^3$ and use the analytic interpolation
\[
\mathcal B_n(z)=\prod_{u=2}^{4}\binom{z+(10+u)n}{10n}
                 \binom{15n}{n+z}\binom{15n}{z}.
\]
The first three factors are polynomials, and the last two are defined using reciprocal gamma functions. At every integer $k$, their values and derivatives are exactly $B_{n,k},B'_{n,k}$ from Lemmas 6.9–6.10 (`prop:z34-q-iv`, `prop:z34-p-iv`). The derivative is that of this interpolation, including where the value is zero.

Gamma reflection gives the meromorphic identity
\[
C'_nF_n(-2n-z)=(-1)^n\frac{\pi^2}{\sin^2(\pi z)}\mathcal B_n(z).
\]
Indeed, the numerator on the left is $\prod_{u=2}^{4}(z+un+1)_{10n}$, and the denominator is $(-n-z)_{16n+1}(-z)_{14n+1}$. Applying reflection to the two denominator factors gives the right side.

Put $z=k+\varepsilon$ and $j=2n+k$. The left Laurent expansion is
$C'_na_j\varepsilon^{-2}-C'_nb_j\varepsilon^{-1}+O(1)$; the right one is
$(-1)^n(B_{n,k}\varepsilon^{-2}+B'_{n,k}\varepsilon^{-1})+O(1)$.
Thus, for every $n\ge0$ and $-n\le k\le15n$,
\[
C'_na_{2n+k}=(-1)^nB_{n,k},\qquad
C'_nb_{2n+k}=-(-1)^nB'_{n,k}.
\]
This includes both simple-pole tails: there $a_{2n+k}=B_{n,k}=0$, while the derivative term remains. Because $j+2n=k+4n$, summation gives
\[
C'_n(Q_n,P_{3,n})=(-1)^n(\mathbf q_n,\mathbf p_{3,n}),\qquad
\frac{P_{3,n}}{Q_n}=\frac{\mathbf p_{3,n}}{\mathbf q_n}=\frac{p_{3,n}}{q_n}.
\]
Here $\mathbf q_n>0$ and $(\mathbf q_n,\mathbf p_{3,n})=(C_n/12)(q_n,p_{3,n})$ by the two cited lemmas. This proves the identification for every index and transfers Section 6's error estimate to the appendix sums.

The next cell independently checks the coefficient formulas and both sums at seven indices, including every simple pole. These are diagnostics of the implementation; the argument above establishes the identity for all $n$. No other notebook is imported or run.

In [4]:
from same_sums import check_same_sums
identity_checks = [check_same_sums(n) for n in range(7)]
for row in identity_checks:
    print(row)
print('All coefficient and ratio checks passed, including both simple-pole tails.')

{'n': 0, 'coefficient_pairs': 1, 'same_ratio': True}
{'n': 1, 'coefficient_pairs': 17, 'same_ratio': True}
{'n': 2, 'coefficient_pairs': 33, 'same_ratio': True}
{'n': 3, 'coefficient_pairs': 49, 'same_ratio': True}
{'n': 4, 'coefficient_pairs': 65, 'same_ratio': True}
{'n': 5, 'coefficient_pairs': 81, 'same_ratio': True}
{'n': 6, 'coefficient_pairs': 97, 'same_ratio': True}
All coefficient and ratio checks passed, including both simple-pole tails.


In [5]:
# Arb interval arithmetic prevents apparent cancellation from being mistaken for accuracy.
ctx.dps = 250
for n in (1, 2, 3, 5):
    Q_n, P3_n, _ = kernel_readout(n)
    error = abs(arb(3).zeta() - arb(P3_n/Q_n))
    assert error > 0
    print(f'n={n}: P3_n/Q_n = {arb(P3_n/Q_n).str(22)}; '
          f'-log(error)/n = {(-error.log()/n).str(12)}')


n=1: P3_n/Q_n = [1.202056903159594285400 +/- 2.62e-22]; -log(error)/n = [111.397149396 +/- 3.20e-10]
n=2: P3_n/Q_n = [1.202056903159594285400 +/- 2.62e-22]; -log(error)/n = [112.313664092 +/- 1.92e-11]
n=3: P3_n/Q_n = [1.202056903159594285400 +/- 2.62e-22]; -log(error)/n = [112.188936848 +/- 2.46e-10]
n=5: P3_n/Q_n = [1.202056903159594285400 +/- 2.62e-22]; -log(error)/n = [112.503476 +/- 5.84e-7]


## 3. Appendix B.4, “The common divisor”: finite integrality checks

Set $d_m=\operatorname{lcm}(1,\ldots,m)$ and
\[
C'_n=\frac{(15n)!^2}{(10n)!^3}.
\]
The baseline statement is $C'_na_j\in\mathbb Z$ and $d_{16n}C'_nb_j\in\mathbb Z$. Since the harmonic indices are at most $19n$, this gives
$6C'_nd_{19n}^3(Q_n,P_{3,n})\in\mathbb Z^2$.

The exact baseline profile, first-order shells, and second-order windows define the appendix divisor $\mathcal D'_n$. For eligible primes $p>29$, $p^2>30n$, and $p\nmid n$, its exponent is the baseline plus the two relevant indicator functions. The next cell uses the bundled exact chamber fixture only to run finite checks. Notebook 02 independently **regenerates all 270 cells** and proves the advertised density calculation from them. The final integer representative is a multiple of the reduced denominator; it need not already be in lowest terms.


In [6]:
reference = json.loads((HERE/'data/chamber_reference.json').read_text())
rows = reference['second_order']
for n in (1, 2, 3, 10, 49, 99):
    print(finite_integer_check(n, rows))


{'n': 1, 'first_order_primes': 0, 'second_order_primes': 0, 'integer_denominator_digits': 52, 'reduced_denominator_digits': 36}
{'n': 2, 'first_order_primes': 0, 'second_order_primes': 0, 'integer_denominator_digits': 97, 'reduced_denominator_digits': 76}
{'n': 3, 'first_order_primes': 1, 'second_order_primes': 0, 'integer_denominator_digits': 136, 'reduced_denominator_digits': 115}
{'n': 10, 'first_order_primes': 8, 'second_order_primes': 1, 'integer_denominator_digits': 415, 'reduced_denominator_digits': 392}


{'n': 49, 'first_order_primes': 30, 'second_order_primes': 4, 'integer_denominator_digits': 2000, 'reduced_denominator_digits': 1955}


{'n': 99, 'first_order_primes': 54, 'second_order_primes': 8, 'integer_denominator_digits': 4004, 'reduced_denominator_digits': 3936}


## 4. Lemma 6.8 (`lem:z34-selected-error`): the spectral rate

Using $S_n=\operatorname{diag}(1,n^{-1},n^{-2},n^{-3})$, compute
$T=\lim_{n\to\infty}S_n^{-1}M(n)S_{n+1}$ directly from the leading terms of the rebuilt matrix. Its characteristic polynomial has the exact factor $X+1$. The remaining cubic has one positive real root $\lambda$ and a complex-conjugate pair of modulus $\rho<1$.

**The unwanted $-1$ mode matters.** The eigenvalue list alone would not justify the error rate for a selected sequence. The all-index ratio identity proved above transfers the joint error estimate without assuming away the native matrix's extra mode. Interpreting the separate native linear-form exponent also uses the paper's recessive-solution/readout argument. Given that analytic input,
\[
\log|Q_n|/n\to\log\lambda,\qquad
\limsup\tfrac1n\log|Q_n\zeta(3)-P_{3,n}|\le\log\rho,
\qquad g=\log\lambda-\log\rho.
\]
The code encloses $\lambda$ between adjacent integers, verifies the cubic's signs and negative discriminant, derives $\rho^2=-c/\lambda$ by Vieta, and evaluates logarithms by a rational atanh series with a rigorous positive tail bound. Displayed floats only summarize the exact rational interval endpoints used by assertions.


In [7]:
T = matrix_at_infinity(M)  # Exact comparison with the supplied reference polynomial.
spectral = spectral_enclosures(T)
for name, (lower, upper) in spectral.items():
    print(f'{name:24s}: [{float(lower):.15f}, {float(upper):.15f}]')
assert spectral['g'][0] > F('112.8075')
assert spectral['raw_Q_growth'][1] < F('53.287584')
assert spectral['scaled_Q_growth'][1] < F('65.451536565866')
print('All assertions passed. Runtime (seconds):', round(time.perf_counter()-started, 2))


raw_Q_growth            : [53.287583322620208, 53.287583322620208]
raw_linear_error        : [-59.519955924834477, -59.519955924834477]
Cprime_growth           : [12.163953243244931, 12.163953243244931]
g                       : [112.807539247454685, 112.807539247454685]
scaled_Q_growth         : [65.451536565865140, 65.451536565865140]
scaled_linear_error     : [-47.356002681589544, -47.356002681589544]
All assertions passed. Runtime (seconds): 8.85


The source paper's $53.287584$ is used as an **upper bound**, and $59.519955$ as a conservative lower bound for linear-form decay; the actual spectral values are approximately $53.2875833226202$ and $59.5199559248345$. The factorial contribution is $30\log(3/2)\approx12.1639532432449$. In the paper's final arithmetic, $12.163953$ is a truncated decimal, **not an upper bound**. Notebook 02 therefore uses the exact certified combined growth bound $65.451536565866$ and also checks the printed final bound $h<91.992990$ directly. This keeps the final theorem correct without asserting a false intermediate inequality.
